<a href="https://colab.research.google.com/github/avi-dot-ai/FL-W/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/avi-dot-ai/FL-W/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*



**Rule:**  Review a page when it is visibly on Page 1 (average position 4–10), has at least 3,000 impressions in the observed 90-day window, and has CTR at or below 0.30%. Rank eligible pages by the estimated additional clicks if CTR reached 0.30%; the score is not a forecast or a promise of lift.

**One action label:** `review_title_snippet_or_intent`  \n
**One reason code:** `visible_page1_low_ctr`

The first audit is tied to FlyRank's CTR-fix logic: CTR should be interpreted with position, not alone. The second checks whether the visibility threshold really separates pages where a small CTR change could matter. Each table prints `n`; verdicts are directional checks for this snapshot, not causal evidence.

In [10]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Define paths using Path objects, relative to the current working directory (REPO_DIR)
DATA_PATH = Path('data/raw/content_refresh_anonymized.csv')
OUTPUT_DIR = Path('work/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
assert df['content_id'].is_unique
print(f'Rows loaded: {len(df):,}; one row per pseudonymized content item.')

# Signal 1 — real FlyRank CTR-fix logic: position provides the CTR context.
position_source = df.loc[df['avg_position'].gt(0)].copy()  # zero means no position data
position_source['position_band'] = pd.cut(
    position_source['avg_position'],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=['top_3', 'page_1_4_10', 'striking_11_20', 'page_3_5', 'deep_50_plus'],
    include_lowest=False,
)
ctr_by_position = (
    position_source.groupby('position_band', observed=False)
    .agg(
        n=('ctr', 'size'),
        median_ctr_pct=('ctr', 'median'),
        low_ctr_at_or_below_0_30_pct=('ctr', lambda s: 100 * s.le(0.30).mean()),
    )
    .round(2)
)
print('Signal 1 — CTR by average-position bucket (flag-linked):')
display(ctr_by_position)
print('Verdict: CONFIRMED — median CTR is materially higher in the Page-1 bucket than in the deeper buckets, so position context is necessary before treating low CTR as a fix opportunity.')

# Signal 2 — exposure: a 0.10 percentage-point CTR change has more operational weight at higher impression volume.
volume_source = df.copy()
volume_source['impression_band'] = pd.cut(
    volume_source['impressions_90d'],
    bins=[0, 299, 2_999, 29_999, np.inf],
    labels=['low_1_299', 'moderate_300_2999', 'good_3000_29999', 'excellent_30000_plus'],
    include_lowest=True,
)
volume_by_impressions = (
    volume_source.groupby('impression_band', observed=False)
    .agg(
        n=('impressions_90d', 'size'),
        median_impressions=('impressions_90d', 'median'),
        median_clicks=('clicks_90d', 'median'),
        mean_clicks_from_0_10pp_ctr=('impressions_90d', lambda s: (s * 0.001).mean()),
    )
    .round(2)
)
print('Signal 2 — exposure by impression bucket:')
display(volume_by_impressions)
print('Verdict: CONFIRMED — the high-volume buckets have much larger click exposure from the same 0.10 percentage-point CTR change, supporting the 3,000-impression review threshold.')

Rows loaded: 30,000; one row per pseudonymized content item.
Signal 1 — CTR by average-position bucket (flag-linked):


,n,median_ctr_pct,low_ctr_at_or_below_0_30_pct
position_band,,,
top_3,1141,0.00,72.66
page_1_4_10,11842,0.16,67.06
striking_11_20,7273,0.10,75.51
page_3_5,7225,0.03,86.69
deep_50_plus,1314,0.00,94.52


Verdict: CONFIRMED — median CTR is materially higher in the Page-1 bucket than in the deeper buckets, so position context is necessary before treating low CTR as a fix opportunity.
Signal 2 — exposure by impression bucket:


,n,median_impressions,median_clicks,mean_clicks_from_0_10pp_ctr
impression_band,,,,
low_1_299,11248,31.0,0.0,0.07
moderate_300_2999,10469,998.0,1.0,1.21
good_3000_29999,7205,7249.0,16.0,9.62
excellent_30000_plus,1078,48675.0,116.0,67.97


Verdict: CONFIRMED — the high-volume buckets have much larger click exposure from the same 0.10 percentage-point CTR change, supporting the 3,000-impression review threshold.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*


The score is `impressions_90d × (0.30 − ctr) / 100` for eligible pages and zero otherwise. Its units are **estimated clicks at a 0.30% CTR reference**, which makes the sort order readable. It uses only snapshot-level page metrics and explicit thresholds—no target, trend field, IDs, product flags, fitted weights, or future window.

In [11]:
MIN_IMPRESSIONS = 3_000
MIN_POSITION = 4
MAX_POSITION = 10
CTR_REFERENCE_PCT = 0.30
ACTION_LABEL = 'review_title_snippet_or_intent'
REASON_CODE = 'visible_page1_low_ctr'

queue = df.copy()
eligible = (
    queue['impressions_90d'].ge(MIN_IMPRESSIONS)
    & queue['avg_position'].between(MIN_POSITION, MAX_POSITION)
    & queue['ctr'].le(CTR_REFERENCE_PCT)
)
queue['score'] = np.where(
    eligible,
    queue['impressions_90d'] * (CTR_REFERENCE_PCT - queue['ctr']) / 100,
    0.0,
)
queue['reason_code'] = np.where(eligible, REASON_CODE, 'not_eligible')
queue['action_label'] = np.where(eligible, ACTION_LABEL, 'no_action')

ranked_queue = (
    queue.loc[eligible, [
        'content_id', 'score', 'reason_code', 'action_label', 'impressions_90d', 'avg_position', 'ctr'
    ]]
    .sort_values(['score', 'impressions_90d'], ascending=[False, False])
    .reset_index(drop=True)
)
ranked_queue.insert(0, 'priority_rank', np.arange(1, len(ranked_queue) + 1))
CSV_PATH = OUTPUT_DIR / 'baseline_action_score.csv'
ranked_queue.to_csv(CSV_PATH, index=False)

receipt = {
    'rows_scored': int(len(queue)),
    'eligible_rows': int(len(ranked_queue)),
    'rule': {
        'min_impressions_90d': MIN_IMPRESSIONS,
        'avg_position_range': [MIN_POSITION, MAX_POSITION],
        'max_ctr_pct': CTR_REFERENCE_PCT,
        'score': 'impressions_90d * (0.30 - ctr) / 100 for eligible pages',
        'reason_code': REASON_CODE,
        'action_label': ACTION_LABEL,
    },
    'signal_verdicts': {'ctr_by_position': 'CONFIRMED', 'impression_exposure': 'CONFIRMED'},
    'leakage_exclusions': ['trend_direction', 'trend_pct', 'is_declining_label', 'content_id', 'client_id'],
}
RECEIPT_PATH = OUTPUT_DIR / 'w04_baseline_score_metrics.json'
RECEIPT_PATH.write_text(json.dumps(receipt, indent=2) + '\n', encoding='utf-8')

print(f'Wrote {len(ranked_queue):,} ranked candidates to {CSV_PATH.resolve()}')
print(f'Wrote run receipt to {RECEIPT_PATH.resolve()}')
display(ranked_queue.head(10))

Wrote 2,354 ranked candidates to /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/work/outputs/baseline_action_score.csv
Wrote run receipt to /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/work/outputs/w04_baseline_score_metrics.json


,priority_rank,content_id,score,reason_code,action_label,impressions_90d,avg_position,ctr
0,1,content_5fe46e04994d,828.3440,visible_page1_low_ctr,review_title_snippet_or_intent,517715,4.2,0.14
1,2,content_36ff89c8214e,737.7425,visible_page1_low_ctr,review_title_snippet_or_intent,295097,7.3,0.05
2,3,content_c8e9d6ab9013,626.0340,visible_page1_low_ctr,review_title_snippet_or_intent,208678,9.7,0.00
3,4,content_c84a0ab98e90,602.8317,visible_page1_low_ctr,review_title_snippet_or_intent,223271,7.8,0.03
4,5,content_cb112fce36be,433.8740,visible_page1_low_ctr,review_title_snippet_or_intent,309910,5.6,0.16
5,6,content_73c54f78c06a,427.9260,visible_page1_low_ctr,review_title_snippet_or_intent,213963,4.7,0.10
6,7,content_453722754fea,406.2291,visible_page1_low_ctr,review_title_snippet_or_intent,140079,7.6,0.01
7,8,content_91652435f57a,383.0160,visible_page1_low_ctr,review_title_snippet_or_intent,159590,7.8,0.06
8,9,content_a7427266c305,382.1109,visible_page1_low_ctr,review_title_snippet_or_intent,201111,5.7,0.11
9,10,content_c1fe78bc4e37,361.9485,visible_page1_low_ctr,review_title_snippet_or_intent,134055,7.5,0.03


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Each line below states the action, why the candidate appears, and what would make the recommendation wrong. Pseudonymous content IDs stay in the CSV for the operator; this review uses priority rank only.

In [12]:
top10 = ranked_queue.head(10).copy()
assert len(top10) == 10, 'The rule should yield at least ten candidates for the required review.'

for row in top10.itertuples(index=False):
    why = (
        f'{int(row.impressions_90d):,} impressions, position {row.avg_position:.1f}, and CTR {row.ctr:.2f}% '
        f'produce an estimated {row.score:.1f}-click gap to the 0.30% reference.'
    )
    wrong = (
        'Wrong if the average position hides a query mix where low CTR is expected, or if SERP layout, brand intent, '
        'or tracking—not the title, snippet, or page intent—caused the click pattern.'
    )
    print(f'#{row.priority_rank} — Action: {row.action_label}. Why: {why} What would make it wrong: {wrong}')

review_table = top10[['priority_rank', 'action_label', 'reason_code', 'impressions_90d', 'avg_position', 'ctr', 'score']].copy()
display(review_table)

#1 — Action: review_title_snippet_or_intent. Why: 517,715 impressions, position 4.2, and CTR 0.14% produce an estimated 828.3-click gap to the 0.30% reference. What would make it wrong: Wrong if the average position hides a query mix where low CTR is expected, or if SERP layout, brand intent, or tracking—not the title, snippet, or page intent—caused the click pattern.
#2 — Action: review_title_snippet_or_intent. Why: 295,097 impressions, position 7.3, and CTR 0.05% produce an estimated 737.7-click gap to the 0.30% reference. What would make it wrong: Wrong if the average position hides a query mix where low CTR is expected, or if SERP layout, brand intent, or tracking—not the title, snippet, or page intent—caused the click pattern.
#3 — Action: review_title_snippet_or_intent. Why: 208,678 impressions, position 9.7, and CTR 0.00% produce an estimated 626.0-click gap to the 0.30% reference. What would make it wrong: Wrong if the average position hides a query mix where low CTR is expecte

,priority_rank,action_label,reason_code,impressions_90d,avg_position,ctr,score
0,1,review_title_snippet_or_intent,visible_page1_low_ctr,517715,4.2,0.14,828.3440
1,2,review_title_snippet_or_intent,visible_page1_low_ctr,295097,7.3,0.05,737.7425
2,3,review_title_snippet_or_intent,visible_page1_low_ctr,208678,9.7,0.00,626.0340
3,4,review_title_snippet_or_intent,visible_page1_low_ctr,223271,7.8,0.03,602.8317
4,5,review_title_snippet_or_intent,visible_page1_low_ctr,309910,5.6,0.16,433.8740
5,6,review_title_snippet_or_intent,visible_page1_low_ctr,213963,4.7,0.10,427.9260
6,7,review_title_snippet_or_intent,visible_page1_low_ctr,140079,7.6,0.01,406.2291
7,8,review_title_snippet_or_intent,visible_page1_low_ctr,159590,7.8,0.06,383.0160
8,9,review_title_snippet_or_intent,visible_page1_low_ctr,201111,5.7,0.11,382.1109
9,10,review_title_snippet_or_intent,visible_page1_low_ctr,134055,7.5,0.03,361.9485


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


The lowest-scored eligible pages are the first candidates I would drop if review capacity is limited. Their Page-1 average may conceal many lower-ranked queries, and the 0.30% reference is deliberately a screening threshold rather than a counterfactual. The check below documents the exact inputs used.

In [13]:
weak_picks = ranked_queue.tail(5).sort_values('priority_rank')
print('Five lowest-scored eligible candidates (weakest operational picks):')
display(weak_picks[['priority_rank', 'impressions_90d', 'avg_position', 'ctr', 'score']])

used_inputs = {'impressions_90d', 'avg_position', 'ctr'}
forbidden_or_label_derived = {'trend_direction', 'trend_pct', 'is_declining_label', 'content_id', 'client_id'}
assert used_inputs.isdisjoint(forbidden_or_label_derived)
assert not any(col.startswith(('impressions_last_', 'clicks_last_', 'sessions_last_')) for col in used_inputs)
print('Leakage check: PASS — score inputs are impressions_90d, avg_position, and ctr only.')
print('No label-derived trend fields, IDs, product flags, or future-window inputs are used.')

Five lowest-scored eligible candidates (weakest operational picks):


,priority_rank,impressions_90d,avg_position,ctr,score
2349,2350,3972,7.2,0.3,0.0
2350,2351,3965,9.3,0.3,0.0
2351,2352,3662,5.8,0.3,0.0
2352,2353,3660,4.7,0.3,0.0
2353,2354,3371,5.9,0.3,0.0


Leakage check: PASS — score inputs are impressions_90d, avg_position, and ctr only.
No label-derived trend fields, IDs, product flags, or future-window inputs are used.


## Self-check


Two bucket tables with n and one-word verdicts are printed; the CTR-versus-position table is tied to FlyRank CTR-fix logic.

One transparent rule creates a score, one reason code, and one action label.

The notebook writes work/outputs/baseline_action_score.csv and a small metrics receipt.

Ten ranked rows receive an action, an evidence-based reason, and a skeptical failure condition.

The score excludes label-derived trend fields, IDs, product flags, and future-window inputs.

Executed top to bottom on the repository's anonymized starter slice.